In [3]:
# """
# 計算利潤並產生報表
# ------------------
# 
# 這個 notebook 支援兩種資料來源：
# 
# 選項 A：使用自己的 Trello 資料（需要有 API 金鑰）
#   訂單檔案 = "data/2026-Trello-data.csv"
#   成本檔案 = "data/Cost.xlsx"
# 
# 選項 B：使用範例資料（不需 API 金鑰，適合測試）
#   訂單檔案 = "sample_data/訂單資料_範例.csv"
#   成本檔案 = "sample_data/成本資料_範例.xlsx"
# 
# 請根據你的情況，修改下面的檔案路徑
# """
import pandas as pd
import re
from datetime import datetime
import os

def merge_and_calculate_profit(trello_csv_path, cost_excel_path):
    # 读取Trello导出的CSV文件
    df_trello = pd.read_csv(trello_csv_path)
    
    # 读取成本Excel文件
    df_cost = pd.read_excel(cost_excel_path)
    
    # 数据预处理
    # 确保金额列是数值类型
    df_trello['Total_amount'] = pd.to_numeric(df_trello['Total_amount'], errors='coerce')
    
    # 保存原始列名用于调试
    print("Trello文件列名:", df_trello.columns.tolist())
    print("成本文件列名:", df_cost.columns.tolist())
    
    # 处理多个产品的情况
    # 检查Product_ID列是否存在
    if 'Product_ID' not in df_trello.columns:
        # 尝试找到可能的替代列名
        possible_names = ['产品ID', 'product_id', '商品ID', '产品编号']
        for name in possible_names:
            if name in df_trello.columns:
                df_trello.rename(columns={name: 'Product_ID'}, inplace=True)
                print(f"已将列 '{name}' 重命名为 'Product_ID'")
                break
        else:
            raise KeyError("找不到产品ID列，请确保数据包含'Product_ID'列")
    
    # 拆分Product_ID列（如果有多个产品）
    df_trello['Product_IDs'] = df_trello['Product_ID'].astype(str).str.split(',')
    
    # 将每个产品拆分为单独的行
    df_exploded = df_trello.explode('Product_IDs')
    df_exploded['Product_IDs'] = df_exploded['Product_IDs'].str.strip()
    
    # 重命名成本文件的列以避免冲突
    df_cost = df_cost.rename(columns={'Product_ID': 'Cost_Product_ID'})
    
    # 合并成本数据
    merged_df = pd.merge(
        df_exploded,
        df_cost,
        left_on='Product_IDs',
        right_on='Cost_Product_ID',
        how='left'
    )
    
    # 计算每个产品的成本
    merged_df['Cost'] = pd.to_numeric(merged_df['Cost'], errors='coerce').fillna(0)
    
    # 创建临时唯一标识符
    merged_df['temp_id'] = merged_df.index
    
    # 检查必要列是否存在
    required_columns = ['List_ID', 'Date', 'Total_amount', 'Phone']
    missing_columns = [col for col in required_columns if col not in merged_df.columns]
    
    if missing_columns:
        raise KeyError(f"缺少必要列: {', '.join(missing_columns)}")
    
    # 聚合数据 - 按原始行（订单）分组
    result_df = merged_df.groupby('temp_id').agg(
        List_ID=('List_ID', 'first'),
        Date=('Date', 'first'),
        Total_amount=('Total_amount', 'first'),
        Phone=('Phone', 'first'),
        Original_Product_ID=('Product_ID', 'first'),  # 原始产品ID
        Total_Cost=('Cost', 'sum'),
        Product_Count=('Product_IDs', 'count')
    ).reset_index()
    
    # 重命名成本列
    result_df = result_df.rename(columns={'Total_Cost': 'Cost'})
    
    # 计算利润
    result_df['Profit'] = result_df['Total_amount'] - result_df['Cost']
    
    # 生成Order_ID
    def generate_order_id(row):
        try:
            # 格式化日期：从YYYY-MM-DD转换为YYYYMMDD
            date_str = str(row['Date'])
            date_part = ''.join(filter(str.isdigit, date_str))[:8]
            
            # 获取List_ID的最后4位
            list_id = str(row['List_ID'])
            list_id_part = list_id[-4:] if len(list_id) >= 4 else list_id.zfill(4)
            
            return f"ORD-V9-{date_part}-{list_id_part}"
        except Exception as e:
            print(f"生成Order_ID时出错: {str(e)}")
            return "ORD-ERROR"
    
    result_df['Order_ID'] = result_df.apply(generate_order_id, axis=1)
    
    # 选择需要的列
    final_df = result_df[['Order_ID', 'Date', 'Total_amount', 'Cost', 'Profit', 'Phone']]
    
    # 生成输出文件名
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"Profit_Report_{timestamp}.csv"
    output_path = os.path.join(os.path.dirname(trello_csv_path), output_filename)
    
    # 保存为CSV
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    return output_path, final_df

if __name__ == "__main__":
    # 文件路径
    trello_csv = r"C:\Users\chine\Desktop\trello_project\分组聚合报表_20250812_132825.csv"
    cost_excel = r"C:\Users\chine\Desktop\trello_project\Cost.xlsx"
    
    try:
        # 处理文件
        output_file, result_df = merge_and_calculate_profit(trello_csv, cost_excel)
        
        print(f"✅ 文件处理完成！输出文件: {output_file}")
        print(f"📊 订单数量: {len(result_df)}")
        
        # 显示前5行预览
        print("\n数据预览:")
        print(result_df.head())
        
        # 保存完整结果供调试
        debug_filename = os.path.join(os.path.dirname(trello_csv), "debug_full_result.csv")
        result_df.to_csv(debug_filename, index=False, encoding='utf-8-sig')
        print(f"🔍 完整调试数据已保存至: {debug_filename}")
        
    except Exception as e:
        print(f"\n❌ 处理过程中出错: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\n💡 建议检查:")
        print("1. 确保输入文件路径正确")
        print("2. 确认Trello导出文件包含'Product_ID'列")
        print("3. 确认成本文件包含'Product_ID'和'Cost'列")
        print("4. 检查列名是否匹配（注意大小写和空格）")
        
        # 尝试显示文件列名
        try:
            df = pd.read_csv(trello_csv, nrows=1)
            print(f"\nTrello文件列名: {', '.join(df.columns)}")
        except:
            pass
        
        try:
            df = pd.read_excel(cost_excel, nrows=1)
            print(f"成本文件列名: {', '.join(df.columns)}")
        except:
            pass

Trello文件列名: ['List_ID', 'List_Name', 'Date', 'Product_ID', 'Total_amount', 'Phone']
成本文件列名: ['Unnamed: 0', 'Product_ID', 'Product_Name', 'Cost', 'profit', 'selling price']
✅ 文件处理完成！输出文件: C:\Users\chine\Desktop\trello_project\Profit_Report_20250812_175849.csv
📊 订单数量: 16

数据预览:
              Order_ID       Date  Total_amount  Cost  Profit  Phone
0  ORD-V9-2025730-040e  2025/7/30        1390.0   0.0  1390.0    NaN
1  ORD-V9-2025810-0a13  2025/8/10        1440.0   0.0  1440.0    NaN
2  ORD-V9-2025830-8570  2025/8/30         148.0   0.0   148.0    NaN
3  ORD-V9-2025731-0518  2025/7/31         123.0   0.0   123.0    NaN
4   ORD-V9-202587-8139   2025/8/7         300.0   0.0   300.0    NaN
🔍 完整调试数据已保存至: C:\Users\chine\Desktop\trello_project\debug_full_result.csv


In [ ]:
以上版本是成功結合二個EXCEL, 但是還是利潤和成本還沒做到.

In [ ]:
1.Cost.xlsx 檔的Product_ID是 seta_tea_1f
2.分组聚合报表_20250812_132825.csv  檔的Product_ID是 seta_tea_1f67
因為我想表逹seta_tea_1f有67份, 要自動替我乘.
明顯二個Product_ID是 對不上的, 那我該如何辦呢?要改那裡才能對得上或者要如何表逹產品的份量, 讓它會自動替我乘? 如果多過2個產品呢, 做法又如何?

In [1]:
import pandas as pd
import re
from datetime import datetime
import os
import numpy as np

def parse_product_id(product_id):
    """
    解析产品ID，提取基础产品ID和数量
    格式：基础产品ID + 数量（数字结尾）
    例如：'seta_tea_1f67' -> ('seta_tea_1f', 67)
    """
    try:
        # 匹配末尾的数字
        match = re.search(r'(\D+?)(\d+)$', product_id)
        if match:
            base_id = match.group(1)
            quantity = int(match.group(2))
            return base_id, quantity
    except:
        pass
    # 如果没有匹配到数量，默认为1
    return product_id, 1

def merge_and_calculate_profit(trello_csv_path, cost_excel_path):
    # 读取Trello导出的CSV文件
    df_trello = pd.read_csv(trello_csv_path)
    
    # 读取成本Excel文件
    df_cost = pd.read_excel(cost_excel_path)
    
    # 数据预处理
    # 确保金额列是数值类型
    df_trello['Total_amount'] = pd.to_numeric(df_trello['Total_amount'], errors='coerce')
    
    print("Trello文件列名:", df_trello.columns.tolist())
    print("成本文件列名:", df_cost.columns.tolist())
    
    # 处理多个产品的情况
    # 检查Product_ID列是否存在
    if 'Product_ID' not in df_trello.columns:
        # 尝试找到可能的替代列名
        possible_names = ['产品ID', 'product_id', '商品ID', '产品编号']
        for name in possible_names:
            if name in df_trello.columns:
                df_trello.rename(columns={name: 'Product_ID'}, inplace=True)
                print(f"已将列 '{name}' 重命名为 'Product_ID'")
                break
        else:
            raise KeyError("找不到产品ID列，请确保数据包含'Product_ID'列")
    
    # 拆分Product_ID列（如果有多个产品）
    df_trello['Product_IDs'] = df_trello['Product_ID'].astype(str).str.split(',')
    
    # 将每个产品拆分为单独的行
    df_exploded = df_trello.explode('Product_IDs')
    df_exploded['Product_IDs'] = df_exploded['Product_IDs'].str.strip()
    
    # 解析产品ID和数量
    parsed_info = df_exploded['Product_IDs'].apply(parse_product_id)
    df_exploded['Base_Product_ID'] = parsed_info.apply(lambda x: x[0])
    df_exploded['Quantity'] = parsed_info.apply(lambda x: x[1])
    
    print("\n解析后的产品示例:")
    print(df_exploded[['Product_IDs', 'Base_Product_ID', 'Quantity']].head())
    
    # 重命名成本文件的列以避免冲突
    df_cost = df_cost.rename(columns={'Product_ID': 'Cost_Product_ID'})
    
    # 合并成本数据
    merged_df = pd.merge(
        df_exploded,
        df_cost,
        left_on='Base_Product_ID',
        right_on='Cost_Product_ID',
        how='left'
    )
    
    # 计算每个产品的总成本（成本 × 数量）
    merged_df['Cost'] = pd.to_numeric(merged_df['Cost'], errors='coerce').fillna(0)
    merged_df['Total_Product_Cost'] = merged_df['Cost'] * merged_df['Quantity']
    
    # 创建临时唯一标识符
    merged_df['temp_id'] = merged_df.index
    
    # 检查必要列是否存在
    required_columns = ['List_ID', 'Date', 'Total_amount', 'Phone']
    missing_columns = [col for col in required_columns if col not in merged_df.columns]
    
    if missing_columns:
        raise KeyError(f"缺少必要列: {', '.join(missing_columns)}")
    
    # 聚合数据 - 按原始行（订单）分组
    result_df = merged_df.groupby('temp_id').agg(
        List_ID=('List_ID', 'first'),
        Date=('Date', 'first'),
        Total_amount=('Total_amount', 'first'),
        Phone=('Phone', 'first'),
        Original_Product_ID=('Product_ID', 'first'),
        Total_Cost=('Total_Product_Cost', 'sum'),
        Product_Count=('Product_IDs', 'count'),
        Total_Quantity=('Quantity', 'sum')
    ).reset_index()
    
    # 重命名成本列
    result_df = result_df.rename(columns={'Total_Cost': 'Cost'})
    
    # 计算利润
    result_df['Profit'] = result_df['Total_amount'] - result_df['Cost']
    
    # 生成Order_ID
    def generate_order_id(row):
        try:
            # 格式化日期：从YYYY-MM-DD转换为YYYYMMDD
            date_str = str(row['Date'])
            date_part = ''.join(filter(str.isdigit, date_str))[:8]
            
            # 获取List_ID的最后4位
            list_id = str(row['List_ID'])
            list_id_part = list_id[-4:] if len(list_id) >= 4 else list_id.zfill(4)
            
            return f"ORD-V9-{date_part}-{list_id_part}"
        except Exception as e:
            print(f"生成Order_ID时出错: {str(e)}")
            return "ORD-ERROR"
    
    result_df['Order_ID'] = result_df.apply(generate_order_id, axis=1)
    
    # 选择需要的列
    final_df = result_df[['Order_ID', 'Date', 'Total_amount', 'Cost', 'Profit', 'Phone']]
    
    # 生成输出文件名
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"Profit_Report_{timestamp}.csv"
    output_path = os.path.join(os.path.dirname(trello_csv_path), output_filename)
    
    # 保存为CSV
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    # 保存详细数据供参考
    detailed_filename = os.path.join(os.path.dirname(trello_csv_path), f"Detailed_Report_{timestamp}.csv")
    result_df.to_csv(detailed_filename, index=False, encoding='utf-8-sig')
    
    return output_path, final_df, result_df

def parse_product_id_debug(product_id):
    """用于调试的解析函数，打印更多信息"""
    print(f"解析产品ID: {product_id}")
    try:
        # 匹配末尾的数字
        match = re.search(r'(\D+?)(\d+)$', product_id)
        if match:
            base_id = match.group(1)
            quantity = int(match.group(2))
            print(f"  解析结果: 基础ID={base_id}, 数量={quantity}")
            return base_id, quantity
    except Exception as e:
        print(f"  解析出错: {str(e)}")
    # 如果没有匹配到数量，默认为1
    print("  未找到数量，默认为1")
    return product_id, 1

if __name__ == "__main__":
    # 文件路径
    trello_csv = r"C:\Users\chine\Desktop\trello_project\分组聚合报表_20250812_132825.csv"
    cost_excel = r"C:\Users\chine\Desktop\trello_project\Cost.xlsx"
    
    try:
        print("="*50)
        print("订单利润计算工具")
        print("="*50)
        
        # 处理文件
        output_file, final_df, detailed_df = merge_and_calculate_profit(trello_csv, cost_excel)
        
        print(f"\n✅ 文件处理完成！输出文件: {output_file}")
        print(f"📊 订单数量: {len(final_df)}")
        
        # 显示前5行预览
        print("\n简要数据预览:")
        print(final_df.head())
        
        # 显示详细数据
        print("\n详细数据预览:")
        print(detailed_df[['Order_ID', 'Date', 'Total_amount', 'Cost', 'Profit', 'Original_Product_ID', 'Total_Quantity']].head())
        
        # 调试解析函数
        test_ids = ["seta_tea_1f67", "coffee_bean_500g12", "milk_tea3", "invalid_id", "no_number"]
        print("\n解析函数测试:")
        for test_id in test_ids:
            base_id, quantity = parse_product_id(test_id)
            print(f"  '{test_id}' -> 基础ID: '{base_id}', 数量: {quantity}")
        
    except Exception as e:
        print(f"\n❌ 处理过程中出错: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\n💡 建议检查:")
        print("1. 确保输入文件路径正确")
        print("2. 确认Trello导出文件包含'Product_ID'列")
        print("3. 确认成本文件包含'Product_ID'和'Cost'列")
        print("4. 检查列名是否匹配（注意大小写和空格）")
        
        # 尝试显示文件列名
        try:
            df = pd.read_csv(trello_csv, nrows=1)
            print(f"\nTrello文件列名: {', '.join(df.columns)}")
        except:
            pass
        
        try:
            df = pd.read_excel(cost_excel, nrows=1)
            print(f"成本文件列名: {', '.join(df.columns)}")
        except:
            pass
        
        # 调试解析函数
        print("\n解析函数调试:")
        test_id = "seta_tea_1f67"
        base_id, quantity = parse_product_id_debug(test_id)
        print(f"  测试ID '{test_id}' -> 基础ID: '{base_id}', 数量: {quantity}")

订单利润计算工具
Trello文件列名: ['List_ID', 'List_Name', 'Date', 'Product_ID', 'Total_amount', 'Phone']
成本文件列名: ['Unnamed: 0', 'Product_ID', 'Product_Name', 'Cost', 'profit', 'selling price']

解析后的产品示例:
       Product_IDs Base_Product_ID  Quantity
0      1c_1c_set67           c_set        67
1  1c_1t_setbox110        t_setbox       110
2    seta_tea_1f10               f        10
3           1_5c14               c        14
4     setc_tea_5c8               c         8

✅ 文件处理完成！输出文件: C:\Users\chine\Desktop\trello_project\Profit_Report_20250812_232748.csv
📊 订单数量: 16

简要数据预览:
              Order_ID       Date  Total_amount  Cost  Profit  Phone
0  ORD-V9-2025730-040e  2025/7/30        1390.0   0.0  1390.0    NaN
1  ORD-V9-2025810-0a13  2025/8/10        1440.0   0.0  1440.0    NaN
2  ORD-V9-2025830-8570  2025/8/30         148.0   0.0   148.0    NaN
3  ORD-V9-2025731-0518  2025/7/31         123.0   0.0   123.0    NaN
4   ORD-V9-202587-8139   2025/8/7         300.0   0.0   300.0    NaN

详细数据预览:
       

In [5]:
import pandas as pd
import re
from datetime import datetime
import os
import numpy as np

def parse_product_id(product_id):
    """
    解析产品ID，提取基础产品ID和数量
    格式：基础产品ID + 数量（数字结尾）
    例如：'seta_tea_1f67' -> ('seta_tea_1f', 67)
    """
    try:
        # 匹配末尾的数字
        match = re.search(r'(\D+?)(\d+)$', product_id)
        if match:
            base_id = match.group(1)
            quantity = int(match.group(2))
            return base_id, quantity
    except:
        pass
    # 如果没有匹配到数量，默认为1
    return product_id, 1

def merge_and_calculate_profit(trello_csv_path, cost_excel_path):
    # 读取Trello导出的CSV文件
    df_trello = pd.read_csv(trello_csv_path)
    
    # 读取成本Excel文件
    df_cost = pd.read_excel(cost_excel_path)
    
    # 数据预处理
    # 确保金额列是数值类型
    df_trello['Total_amount'] = pd.to_numeric(df_trello['Total_amount'], errors='coerce').fillna(0)
    
    print("Trello文件列名:", df_trello.columns.tolist())
    print("成本文件列名:", df_cost.columns.tolist())
    
    # 处理多个产品的情况
    # 检查Product_ID列是否存在
    if 'Product_ID' not in df_trello.columns:
        # 尝试找到可能的替代列名
        possible_names = ['产品ID', 'product_id', '商品ID', '产品编号']
        for name in possible_names:
            if name in df_trello.columns:
                df_trello.rename(columns={name: 'Product_ID'}, inplace=True)
                print(f"已将列 '{name}' 重命名为 'Product_ID'")
                break
        else:
            raise KeyError("找不到产品ID列，请确保数据包含'Product_ID'列")
    
    # 创建原始订单的唯一标识符 - 使用索引
    df_trello['order_id_temp'] = df_trello.index
    
    # 拆分Product_ID列（如果有多个产品）
    df_trello['Product_IDs'] = df_trello['Product_ID'].astype(str).str.split(',')
    
    # 将每个产品拆分为单独的行
    df_exploded = df_trello.explode('Product_IDs')
    df_exploded['Product_IDs'] = df_exploded['Product_IDs'].str.strip()
    
    # 解析产品ID和数量
    parsed_info = df_exploded['Product_IDs'].apply(parse_product_id)
    df_exploded['Base_Product_ID'] = parsed_info.apply(lambda x: x[0])
    df_exploded['Quantity'] = parsed_info.apply(lambda x: x[1])
    
    print("\n解析后的产品示例:")
    print(df_exploded[['Product_IDs', 'Base_Product_ID', 'Quantity']].head())
    
    # 重命名成本文件的列以避免冲突
    df_cost = df_cost.rename(columns={'Product_ID': 'Cost_Product_ID'})
    
    # 合并成本数据
    merged_df = pd.merge(
        df_exploded,
        df_cost,
        left_on='Base_Product_ID',
        right_on='Cost_Product_ID',
        how='left'
    )
    
    # 计算每个产品的总成本（成本 × 数量）
    merged_df['Cost'] = pd.to_numeric(merged_df['Cost'], errors='coerce').fillna(0)
    merged_df['Total_Product_Cost'] = merged_df['Cost'] * merged_df['Quantity']
    
    # 检查必要列是否存在
    required_columns = ['List_ID', 'Date', 'Total_amount', 'Phone', 'order_id_temp']
    missing_columns = [col for col in required_columns if col not in merged_df.columns]
    
    if missing_columns:
        raise KeyError(f"缺少必要列: {', '.join(missing_columns)}")
    
    # 聚合数据 - 按原始订单分组
    # 使用 order_id_temp 分组，确保每个原始订单只计算一次
    result_df = merged_df.groupby('order_id_temp').agg(
        List_ID=('List_ID', 'first'),
        Date=('Date', 'first'),
        Total_amount=('Total_amount', 'first'),
        Phone=('Phone', 'first'),
        Original_Product_ID=('Product_ID', 'first'),
        Total_Cost=('Total_Product_Cost', 'sum'),  # 总成本 = 所有产品的(成本×数量)之和
        Total_Quantity=('Quantity', 'sum')  # 总数量 = 所有产品数量之和
    ).reset_index()
    
    # 计算利润
    result_df['Profit'] = result_df['Total_amount'] - result_df['Total_Cost']
    
    # 生成Order_ID
    def generate_order_id(row):
        try:
            # 格式化日期：从YYYY-MM-DD转换为YYYYMMDD
            date_str = str(row['Date'])
            date_part = ''.join(filter(str.isdigit, date_str))[:8]
            
            # 获取List_ID的最后4位
            list_id = str(row['List_ID'])
            list_id_part = list_id[-4:] if len(list_id) >= 4 else list_id.zfill(4)
            
            return f"ORD-V9-{date_part}-{list_id_part}"
        except Exception as e:
            print(f"生成Order_ID时出错: {str(e)}")
            return "ORD-ERROR"
    
    result_df['Order_ID'] = result_df.apply(generate_order_id, axis=1)
    
    # 选择需要的列
    final_df = result_df[['Order_ID', 'Date', 'Total_amount', 'Total_Cost', 'Profit', 'Phone']]
    
    # 重命名成本列
    final_df = final_df.rename(columns={'Total_Cost': 'Cost'})
    
    # 生成输出文件名
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"Profit_Report_{timestamp}.csv"
    output_path = os.path.join(os.path.dirname(trello_csv_path), output_filename)
    
    # 保存为CSV
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    # 保存详细数据供参考
    detailed_filename = os.path.join(os.path.dirname(trello_csv_path), f"Detailed_Report_{timestamp}.csv")
    result_df.to_csv(detailed_filename, index=False, encoding='utf-8-sig')
    
    return output_path, final_df, result_df

def parse_product_id_debug(product_id):
    """用于调试的解析函数，打印更多信息"""
    print(f"解析产品ID: {product_id}")
    try:
        # 匹配末尾的数字
        match = re.search(r'(\D+?)(\d+)$', product_id)
        if match:
            base_id = match.group(1)
            quantity = int(match.group(2))
            print(f"  解析结果: 基础ID={base_id}, 数量={quantity}")
            return base_id, quantity
    except Exception as e:
        print(f"  解析出错: {str(e)}")
    # 如果没有匹配到数量，默认为1
    print("  未找到数量，默认为1")
    return product_id, 1

if __name__ == "__main__":
    # 文件路径
    trello_csv = r"C:\Users\chine\Desktop\trello_project\分组聚合报表_20250812_132825.csv"
    cost_excel = r"C:\Users\chine\Desktop\trello_project\Cost.xlsx"
    
    try:
        print("="*50)
        print("订单利润计算工具")
        print("="*50)
        
        # 处理文件
        output_file, final_df, detailed_df = merge_and_calculate_profit(trello_csv, cost_excel)
        
        print(f"\n✅ 文件处理完成！输出文件: {output_file}")
        print(f"📊 订单数量: {len(final_df)}")
        
        # 显示前5行预览
        print("\n简要数据预览:")
        print(final_df.head())
        
        # 显示详细数据
        print("\n详细数据预览:")
        print(detailed_df[['Order_ID', 'Date', 'Total_amount', 'Total_Cost', 'Profit', 'Original_Product_ID', 'Total_Quantity']].head())
        
        # 计算总利润
        total_profit = detailed_df['Profit'].sum()
        total_revenue = detailed_df['Total_amount'].sum()
        total_cost = detailed_df['Total_Cost'].sum()
        
        print("\n汇总统计:")
        print(f"总收入: ¥{total_revenue:.2f}")
        print(f"总成本: ¥{total_cost:.2f}")
        print(f"总利润: ¥{total_profit:.2f}")
        print(f"平均利润率: {(total_profit/total_revenue*100 if total_revenue != 0 else 0):.2f}%")
        
        # 调试解析函数
        test_ids = ["seta_tea_1f67", "coffee_bean_500g12", "milk_tea3", "invalid_id", "no_number"]
        print("\n解析函数测试:")
        for test_id in test_ids:
            base_id, quantity = parse_product_id(test_id)
            print(f"  '{test_id}' -> 基础ID: '{base_id}', 数量: {quantity}")
        
    except Exception as e:
        print(f"\n❌ 处理过程中出错: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\n💡 建议检查:")
        print("1. 确保输入文件路径正确")
        print("2. 确认Trello导出文件包含'Product_ID'列")
        print("3. 确认成本文件包含'Product_ID'和'Cost'列")
        print("4. 检查列名是否匹配（注意大小写和空格）")
        
        # 尝试显示文件列名
        try:
            df = pd.read_csv(trello_csv, nrows=1)
            print(f"\nTrello文件列名: {', '.join(df.columns)}")
        except:
            pass
        
        try:
            df = pd.read_excel(cost_excel, nrows=1)
            print(f"成本文件列名: {', '.join(df.columns)}")
        except:
            pass
        
        # 调试解析函数
        print("\n解析函数调试:")
        test_id = "seta_tea_1f67"
        base_id, quantity = parse_product_id_debug(test_id)
        print(f"  测试ID '{test_id}' -> 基础ID: '{base_id}', 数量: {quantity}")

订单利润计算工具
Trello文件列名: ['List_ID', 'List_Name', 'Date', 'Product_ID', 'Total_amount', 'Phone']
成本文件列名: ['Unnamed: 0', 'Product_ID', 'Product_Name', 'Cost', 'profit', 'selling price']

解析后的产品示例:
       Product_IDs Base_Product_ID  Quantity
0      1c_1c_set67           c_set        67
1  1c_1t_setbox110        t_setbox       110
2    seta_tea_1f10               f        10
3           1_5c14               c        14
4     setc_tea_5c8               c         8

✅ 文件处理完成！输出文件: C:\Users\chine\Desktop\trello_project\Profit_Report_20250812_235106.csv
📊 订单数量: 12

简要数据预览:
              Order_ID       Date  Total_amount  Cost  Profit  Phone
0  ORD-V9-2025730-040e  2025/7/30        1390.0   0.0  1390.0    NaN
1  ORD-V9-2025810-0a13  2025/8/10        1440.0   0.0  1440.0    NaN
2  ORD-V9-2025830-8570  2025/8/30         148.0   0.0   148.0    NaN
3  ORD-V9-2025731-0518  2025/7/31         123.0   0.0   123.0    NaN
4   ORD-V9-202587-8139   2025/8/7         300.0  36.0   264.0    NaN

详细数据预览:
       

In [6]:
import pandas as pd
import re
from datetime import datetime
import os
import numpy as np
from collections import defaultdict

def parse_product_id(product_id):
    """
    更健壮的产品ID解析函数
    支持多种格式：基础产品ID+数量、基础产品ID(数量)、基础产品ID*数量
    例如：'seta_tea_1f67', 'seta_tea_1f(67)', 'seta_tea_1f*67'
    """
    product_id = str(product_id).strip()
    
    # 尝试匹配数字后缀格式
    match = re.search(r'^(.+?)(\d+)$', product_id)
    if match:
        base_id = match.group(1).rstrip('_')  # 移除末尾可能的下划线
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配括号格式
    match = re.search(r'^(.+?)\((\d+)\)$', product_id)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配乘号格式
    match = re.search(r'^(.+?)\*(\d+)$', product_id)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配x格式
    match = re.search(r'^(.+?)x(\d+)$', product_id, re.IGNORECASE)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 默认返回产品ID和数量1
    return product_id, 1

def merge_and_calculate_profit(trello_csv_path, cost_excel_path):
    # 读取Trello导出的CSV文件
    df_trello = pd.read_csv(trello_csv_path)
    
    # 读取成本Excel文件
    df_cost = pd.read_excel(cost_excel_path)
    
    # 打印初始数据预览
    print("\nTrello数据预览:")
    print(df_trello.head())
    print("\n成本数据预览:")
    print(df_cost.head())
    
    # 数据预处理
    # 确保金额列是数值类型
    df_trello['Total_amount'] = pd.to_numeric(df_trello['Total_amount'], errors='coerce').fillna(0)
    
    print("\nTrello文件列名:", df_trello.columns.tolist())
    print("成本文件列名:", df_cost.columns.tolist())
    
    # 处理多个产品的情况
    # 检查Product_ID列是否存在
    if 'Product_ID' not in df_trello.columns:
        # 尝试找到可能的替代列名
        possible_names = ['产品ID', 'product_id', '商品ID', '产品编号']
        for name in possible_names:
            if name in df_trello.columns:
                df_trello.rename(columns={name: 'Product_ID'}, inplace=True)
                print(f"已将列 '{name}' 重命名为 'Product_ID'")
                break
        else:
            raise KeyError("找不到产品ID列，请确保数据包含'Product_ID'列")
    
    # 创建原始订单的唯一标识符 - 使用索引
    df_trello['order_id_temp'] = df_trello.index
    
    # 拆分Product_ID列（如果有多个产品）
    df_trello['Product_IDs'] = df_trello['Product_ID'].astype(str).str.split(',')
    
    # 将每个产品拆分为单独的行
    df_exploded = df_trello.explode('Product_IDs')
    df_exploded['Product_IDs'] = df_exploded['Product_IDs'].str.strip()
    
    # 解析产品ID和数量
    print("\n开始解析产品ID和数量...")
    parsed_info = df_exploded['Product_IDs'].apply(parse_product_id)
    df_exploded['Base_Product_ID'] = parsed_info.apply(lambda x: x[0])
    df_exploded['Quantity'] = parsed_info.apply(lambda x: x[1])
    
    # 创建成本映射字典
    cost_map = defaultdict(float)
    for _, row in df_cost.iterrows():
        cost_map[str(row['Product_ID']).strip()] = float(row['Cost'])
    
    # 计算每个产品的总成本
    print("\n计算每个产品的成本...")
    df_exploded['Product_Cost'] = df_exploded.apply(lambda row: 
        cost_map.get(row['Base_Product_ID'], 0) * row['Quantity'], axis=1)
    
    # 统计未匹配的产品
    unmatched_products = df_exploded[df_exploded['Product_Cost'] == 0]['Base_Product_ID'].unique()
    if len(unmatched_products) > 0:
        print(f"\n⚠️ 警告: {len(unmatched_products)} 种产品未匹配到成本:")
        for product in unmatched_products[:10]:  # 最多显示10个
            print(f"  - '{product}'")
        print("这些产品的成本将被设为0")
    
    # 按原始订单分组计算总成本
    print("\n按订单分组计算总成本...")
    order_costs = df_exploded.groupby('order_id_temp')['Product_Cost'].sum().reset_index()
    order_costs.rename(columns={'Product_Cost': 'Total_Cost'}, inplace=True)
    
    # 合并回原始数据
    result_df = pd.merge(
        df_trello.drop(columns=['Product_IDs']),
        order_costs,
        on='order_id_temp',
        how='left'
    )
    
    # 填充缺失的成本为0
    result_df['Total_Cost'] = result_df['Total_Cost'].fillna(0)
    
    # 计算利润
    result_df['Profit'] = result_df['Total_amount'] - result_df['Total_Cost']
    
    # 生成Order_ID
    def generate_order_id(row):
        try:
            # 格式化日期：从YYYY-MM-DD转换为YYYYMMDD
            date_str = str(row['Date'])
            date_part = ''.join(filter(str.isdigit, date_str))[:8]
            
            # 获取List_ID的最后4位
            list_id = str(row['List_ID'])
            list_id_part = list_id[-4:] if len(list_id) >= 4 else list_id.zfill(4)
            
            return f"ORD-V9-{date_part}-{list_id_part}"
        except Exception as e:
            print(f"生成Order_ID时出错: {str(e)}")
            return "ORD-ERROR"
    
    print("\n生成订单ID...")
    result_df['Order_ID'] = result_df.apply(generate_order_id, axis=1)
    
    # 选择需要的列
    final_df = result_df[['Order_ID', 'Date', 'Total_amount', 'Total_Cost', 'Profit', 'Phone']]
    
    # 重命名成本列
    final_df = final_df.rename(columns={'Total_Cost': 'Cost'})
    
    # 生成输出文件名
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"Profit_Report_{timestamp}.csv"
    output_path = os.path.join(os.path.dirname(trello_csv_path), output_filename)
    
    # 保存为CSV
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    # 保存详细数据供参考
    detailed_filename = os.path.join(os.path.dirname(trello_csv_path), f"Detailed_Report_{timestamp}.csv")
    result_df.to_csv(detailed_filename, index=False, encoding='utf-8-sig')
    
    return output_path, final_df, result_df

def test_parse_product_id():
    """测试解析函数"""
    test_cases = [
        ("seta_tea_1f67", ("seta_tea_1f", 67)),
        ("coffee_bean_500g12", ("coffee_bean_500g", 12)),
        ("milk_tea(3)", ("milk_tea", 3)),
        ("green_tea*5", ("green_tea", 5)),
        ("black_tea x 10", ("black_tea", 10)),
        ("single_product", ("single_product", 1)),
        ("product_with_number123", ("product_with_number", 123)),
        ("product_with_underscore_456", ("product_with_underscore", 456)),
        ("product-with-dash-789", ("product-with-dash", 789)),
        ("special!product@99", ("special!product@", 99)),
    ]
    
    print("\n解析函数测试:")
    for input_id, expected in test_cases:
        result = parse_product_id(input_id)
        status = "✓" if result == expected else "✗"
        print(f"  {status} 输入: '{input_id}'")
        print(f"    期望: {expected}")
        print(f"    实际: {result}")
        print()

if __name__ == "__main__":
    # 文件路径
    trello_csv = r"C:\Users\chine\Desktop\trello_project\分组聚合报表_20250812_132825.csv"
    cost_excel = r"C:\Users\chine\Desktop\trello_project\Cost.xlsx"
    
    try:
        print("="*50)
        print("订单利润计算工具 (修复版)")
        print("="*50)
        
        # 运行解析函数测试
        test_parse_product_id()
        
        # 处理文件
        output_file, final_df, detailed_df = merge_and_calculate_profit(trello_csv, cost_excel)
        
        print(f"\n✅ 文件处理完成！输出文件: {output_file}")
        print(f"📊 订单数量: {len(final_df)}")
        
        # 显示前5行预览
        print("\n简要数据预览:")
        print(final_df.head())
        
        # 显示详细数据
        print("\n详细数据预览:")
        print(detailed_df[['Order_ID', 'Date', 'Total_amount', 'Total_Cost', 'Profit', 'Product_ID']].head())
        
        # 计算总利润
        total_profit = detailed_df['Profit'].sum()
        total_revenue = detailed_df['Total_amount'].sum()
        total_cost = detailed_df['Total_Cost'].sum()
        
        print("\n汇总统计:")
        print(f"总收入: ¥{total_revenue:.2f}")
        print(f"总成本: ¥{total_cost:.2f}")
        print(f"总利润: ¥{total_profit:.2f}")
        print(f"平均利润率: {(total_profit/total_revenue*100 if total_revenue != 0 else 0):.2f}%")
        
        # 验证数据质量
        zero_cost_orders = detailed_df[detailed_df['Total_Cost'] == 0]
        if len(zero_cost_orders) > 0:
            print(f"\n⚠️ 警告: {len(zero_cost_orders)} 笔订单的成本为0")
            print("可能原因:")
            print("1. 产品ID未匹配到成本数据")
            print("2. 产品数量解析失败")
            print("3. 成本数据缺失")
            print("前5笔零成本订单:")
            print(zero_cost_orders[['Order_ID', 'Product_ID']].head())
        
        negative_profit_orders = detailed_df[detailed_df['Profit'] < 0]
        if len(negative_profit_orders) > 0:
            print(f"\n⚠️ 警告: {len(negative_profit_orders)} 笔订单的利润为负")
            print("前5笔负利润订单:")
            print(negative_profit_orders[['Order_ID', 'Total_amount', 'Total_Cost', 'Profit']].head())
        
    except Exception as e:
        print(f"\n❌ 处理过程中出错: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\n💡 建议检查:")
        print("1. 确保输入文件路径正确")
        print("2. 确认Trello导出文件包含'Product_ID'列")
        print("3. 确认成本文件包含'Product_ID'和'Cost'列")
        print("4. 检查列名是否匹配（注意大小写和空格）")
        
        # 尝试显示文件列名
        try:
            df = pd.read_csv(trello_csv, nrows=1)
            print(f"\nTrello文件列名: {', '.join(df.columns)}")
        except:
            pass
        
        try:
            df = pd.read_excel(cost_excel, nrows=1)
            print(f"成本文件列名: {', '.join(df.columns)}")
        except:
            pass

订单利润计算工具 (修复版)

解析函数测试:
  ✓ 输入: 'seta_tea_1f67'
    期望: ('seta_tea_1f', 67)
    实际: ('seta_tea_1f', 67)

  ✓ 输入: 'coffee_bean_500g12'
    期望: ('coffee_bean_500g', 12)
    实际: ('coffee_bean_500g', 12)

  ✓ 输入: 'milk_tea(3)'
    期望: ('milk_tea', 3)
    实际: ('milk_tea', 3)

  ✗ 输入: 'green_tea*5'
    期望: ('green_tea', 5)
    实际: ('green_tea*', 5)

  ✗ 输入: 'black_tea x 10'
    期望: ('black_tea', 10)
    实际: ('black_tea x ', 10)

  ✓ 输入: 'single_product'
    期望: ('single_product', 1)
    实际: ('single_product', 1)

  ✓ 输入: 'product_with_number123'
    期望: ('product_with_number', 123)
    实际: ('product_with_number', 123)

  ✓ 输入: 'product_with_underscore_456'
    期望: ('product_with_underscore', 456)
    实际: ('product_with_underscore', 456)

  ✗ 输入: 'product-with-dash-789'
    期望: ('product-with-dash', 789)
    实际: ('product-with-dash-', 789)

  ✓ 输入: 'special!product@99'
    期望: ('special!product@', 99)
    实际: ('special!product@', 99)


Trello数据预览:
                    List_ID           List_Na

In [2]:
import pandas as pd
import re
from datetime import datetime
import os
import numpy as np
from collections import defaultdict
import unicodedata

def normalize_string(s):
    """标准化字符串：移除特殊字符、转换为小写、移除空格"""
    if not isinstance(s, str):
        s = str(s)
    # 移除重音符号
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    # 转换为小写
    s = s.lower()
    # 移除所有非字母数字字符（除了下划线和连字符）
    s = re.sub(r'[^\w\s-]', '', s)
    # 替换多个空格为单个空格
    s = re.sub(r'\s+', ' ', s)
    # 移除首尾空格
    s = s.strip()
    # 替换空格为下划线
    s = re.sub(r'\s', '_', s)
    return s

def parse_product_id(product_id):
    """
    更健壮的产品ID解析函数
    支持多种格式：基础产品ID+数量、基础产品ID(数量)、基础产品ID*数量
    例如：'seta_tea_1f67', 'seta_tea_1f(67)', 'seta_tea_1f*67'
    """
    original_id = str(product_id).strip()
    if not original_id:
        return original_id, 1
    
    # 尝试匹配数字后缀格式
    match = re.search(r'^(.+?)(\d{1,10})$', original_id)
    if match:
        base_id = match.group(1).rstrip('_')  # 移除末尾可能的下划线
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配括号格式
    match = re.search(r'^(.+?)\((\d{1,10})\)$', original_id)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配乘号格式
    match = re.search(r'^(.+?)\*(\d{1,10})$', original_id)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配x格式
    match = re.search(r'^(.+?)[x×](\d{1,10})$', original_id, re.IGNORECASE)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 默认返回产品ID和数量1
    return original_id, 1

def merge_and_calculate_profit(trello_csv_path, cost_excel_path):
    # 读取Trello导出的CSV文件
    print(f"读取Trello数据: {trello_csv_path}")
    df_trello = pd.read_csv(trello_csv_path)
    
    # 读取成本Excel文件
    print(f"读取成本数据: {cost_excel_path}")
    df_cost = pd.read_excel(cost_excel_path)
    
    # 打印初始数据预览
    print("\nTrello数据预览:")
    print(df_trello.head(3))
    print("\n成本数据预览:")
    print(df_cost.head(3))
    
    # 数据预处理
    # 确保金额列是数值类型
    df_trello['Total_amount'] = pd.to_numeric(df_trello['Total_amount'], errors='coerce').fillna(0)
    
    print("\nTrello文件列名:", df_trello.columns.tolist())
    print("成本文件列名:", df_cost.columns.tolist())
    
    # 处理多个产品的情况
    # 检查Product_ID列是否存在
    if 'Product_ID' not in df_trello.columns:
        # 尝试找到可能的替代列名
        possible_names = ['产品ID', 'product_id', '商品ID', '产品编号', 'productid', '商品编号']
        for name in possible_names:
            if name in df_trello.columns:
                df_trello.rename(columns={name: 'Product_ID'}, inplace=True)
                print(f"已将列 '{name}' 重命名为 'Product_ID'")
                break
        else:
            raise KeyError("找不到产品ID列，请确保数据包含'Product_ID'列")
    
    # 标准化产品ID列名
    if 'Product_ID' not in df_cost.columns:
        # 尝试找到可能的替代列名
        possible_names = ['产品ID', 'product_id', '商品ID', '产品编号', 'productid', '商品编号']
        for name in possible_names:
            if name in df_cost.columns:
                df_cost.rename(columns={name: 'Product_ID'}, inplace=True)
                print(f"已将成本文件列 '{name}' 重命名为 'Product_ID'")
                break
        else:
            raise KeyError("成本文件中找不到产品ID列")
    
    # 创建原始订单的唯一标识符 - 使用索引
    df_trello['order_id_temp'] = df_trello.index
    
    # 拆分Product_ID列（如果有多个产品）
    df_trello['Product_IDs'] = df_trello['Product_ID'].astype(str).str.split(',')
    
    # 将每个产品拆分为单独的行
    df_exploded = df_trello.explode('Product_IDs')
    df_exploded['Product_IDs'] = df_exploded['Product_IDs'].str.strip()
    
    # 解析产品ID和数量
    print("\n开始解析产品ID和数量...")
    parsed_info = df_exploded['Product_IDs'].apply(parse_product_id)
    df_exploded['Base_Product_ID'] = parsed_info.apply(lambda x: x[0])
    df_exploded['Quantity'] = parsed_info.apply(lambda x: x[1])
    
    # 标准化所有ID以进行匹配
    print("\n标准化产品ID以进行匹配...")
    df_exploded['Base_Product_ID_Norm'] = df_exploded['Base_Product_ID'].apply(normalize_string)
    df_cost['Product_ID_Norm'] = df_cost['Product_ID'].apply(normalize_string)
    
    # 创建成本映射字典
    cost_map = defaultdict(float)
    for _, row in df_cost.iterrows():
        norm_id = row['Product_ID_Norm']
        cost = float(row['Cost']) if not pd.isna(row['Cost']) else 0
        cost_map[norm_id] = cost
    
    # 计算每个产品的总成本
    print("\n计算每个产品的成本...")
    df_exploded['Product_Cost'] = df_exploded.apply(lambda row: 
        cost_map.get(row['Base_Product_ID_Norm'], 0) * row['Quantity'], axis=1)
    
    # 统计未匹配的产品
    unmatched_df = df_exploded[df_exploded['Product_Cost'] == 0]
    unmatched_products = unmatched_df['Base_Product_ID'].unique()
    
    if len(unmatched_products) > 0:
        print(f"\n⚠️ 警告: {len(unmatched_products)} 种产品未匹配到成本:")
        print(f"未匹配产品示例 ({min(5, len(unmatched_products))} of {len(unmatched_products)}):")
        for product in unmatched_products[:5]:
            norm_product = normalize_string(product)
            print(f"  - 原始ID: '{product}'")
            print(f"    标准化ID: '{norm_product}'")
            print(f"    在成本文件中匹配: {'是' if norm_product in cost_map else '否'}")
            
            # 查找最相似的ID
            similarities = []
            for cost_id in cost_map.keys():
                if norm_product in cost_id or cost_id in norm_product:
                    similarity = len(set(norm_product) & set(cost_id)) / max(len(norm_product), len(cost_id))
                    similarities.append((cost_id, similarity))
            
            if similarities:
                best_match = max(similarities, key=lambda x: x[1])
                print(f"    最相似的成本ID: '{best_match[0]}' (相似度: {best_match[1]:.2f})")
            else:
                print("    没有找到相似的ID")
    
    # 按原始订单分组计算总成本
    print("\n按订单分组计算总成本...")
    order_costs = df_exploded.groupby('order_id_temp')['Product_Cost'].sum().reset_index()
    order_costs.rename(columns={'Product_Cost': 'Total_Cost'}, inplace=True)
    
    # 合并回原始数据
    result_df = pd.merge(
        df_trello.drop(columns=['Product_IDs']),
        order_costs,
        on='order_id_temp',
        how='left'
    )
    
    # 填充缺失的成本为0
    result_df['Total_Cost'] = result_df['Total_Cost'].fillna(0)
    
    # 计算利润
    result_df['Profit'] = result_df['Total_amount'] - result_df['Total_Cost']
    
    # 生成Order_ID
    def generate_order_id(row):
        try:
            # 格式化日期：从YYYY-MM-DD转换为YYYYMMDD
            date_str = str(row['Date'])
            date_part = ''.join(filter(str.isdigit, date_str))[:8]
            
            # 获取List_ID的最后4位
            list_id = str(row['List_ID'])
            list_id_part = list_id[-4:] if len(list_id) >= 4 else list_id.zfill(4)
            
            return f"ORD-V9-{date_part}-{list_id_part}"
        except Exception as e:
            print(f"生成Order_ID时出错: {str(e)}")
            return "ORD-ERROR"
    
    print("\n生成订单ID...")
    result_df['Order_ID'] = result_df.apply(generate_order_id, axis=1)
    
    # 选择需要的列
    final_columns = ['Order_ID', 'Date', 'Total_amount', 'Total_Cost', 'Profit', 'Phone']
    
    # 确保所有列都存在
    available_columns = [col for col in final_columns if col in result_df.columns]
    missing_columns = set(final_columns) - set(available_columns)
    
    if missing_columns:
        print(f"⚠️ 缺少列: {', '.join(missing_columns)}")
        # 添加缺失列
        for col in missing_columns:
            result_df[col] = None
    
    final_df = result_df[final_columns]
    
    # 重命名成本列
    final_df = final_df.rename(columns={'Total_Cost': 'Cost'})
    
    # 生成输出文件名
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"Profit_Report_{timestamp}.csv"
    output_path = os.path.join(os.path.dirname(trello_csv_path), output_filename)
    
    # 保存为CSV
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"✅ 利润报告已保存至: {output_path}")
    
    # 保存详细数据供参考
    detailed_filename = os.path.join(os.path.dirname(trello_csv_path), f"Detailed_Report_{timestamp}.csv")
    result_df.to_csv(detailed_filename, index=False, encoding='utf-8-sig')
    print(f"✅ 详细报告已保存至: {detailed_filename}")
    
    # 保存未匹配产品数据
    if not unmatched_df.empty:
        unmatched_filename = os.path.join(os.path.dirname(trello_csv_path), f"Unmatched_Products_{timestamp}.csv")
        unmatched_df[['order_id_temp', 'Product_IDs', 'Base_Product_ID', 'Base_Product_ID_Norm', 'Quantity']].to_csv(unmatched_filename, index=False)
        print(f"⚠️ 未匹配产品数据已保存至: {unmatched_filename}")
    
    return output_path, final_df, result_df

def test_normalization():
    """测试标准化函数"""
    test_cases = [
        ("Seta_Tea-1F", "seta_tea_1f"),
        ("Green Tea (Premium)", "green_tea_premium"),
        ("Milk_Tea*3", "milk_tea_3"),
        ("Black Tea - 500g", "black_tea_500g"),
        ("Café au Lait", "cafe_au_lait"),
        ("产品编号_123", "chan_pin_bian_hao_123"),
        ("SET ITEM 1", "set_item_1"),
    ]
    
    print("\n标准化函数测试:")
    for input_str, expected in test_cases:
        result = normalize_string(input_str)
        status = "✓" if result == expected else "✗"
        print(f"  {status} 输入: '{input_str}'")
        print(f"    期望: '{expected}'")
        print(f"    实际: '{result}'")
        print()

if __name__ == "__main__":
    # 文件路径
    trello_csv = r"C:\Users\chine\Desktop\trello_project\分组聚合报表_20250812_132825.csv"
    cost_excel = r"C:\Users\chine\Desktop\trello_project\Cost.xlsx"
    
    try:
        print("="*50)
        print("订单利润计算工具 (终极修复版)")
        print("="*50)
        
        # 运行标准化函数测试
        test_normalization()
        
        # 处理文件
        output_file, final_df, detailed_df = merge_and_calculate_profit(trello_csv, cost_excel)
        
        print(f"\n✅ 文件处理完成！输出文件: {output_file}")
        print(f"📊 订单数量: {len(final_df)}")
        
        # 显示前5行预览
        print("\n简要数据预览:")
        print(final_df.head())
        
        # 显示详细数据
        print("\n详细数据预览:")
        print(detailed_df[['Order_ID', 'Date', 'Total_amount', 'Total_Cost', 'Profit', 'Product_ID']].head())
        
        # 计算总利润
        total_profit = detailed_df['Profit'].sum()
        total_revenue = detailed_df['Total_amount'].sum()
        total_cost = detailed_df['Total_Cost'].sum()
        
        print("\n汇总统计:")
        print(f"总收入: ¥{total_revenue:.2f}")
        print(f"总成本: ¥{total_cost:.2f}")
        print(f"总利润: ¥{total_profit:.2f}")
        print(f"平均利润率: {(total_profit/total_revenue*100 if total_revenue != 0 else 0):.2f}%")
        
        # 验证数据质量
        zero_cost_orders = detailed_df[detailed_df['Total_Cost'] == 0]
        if len(zero_cost_orders) > 0:
            print(f"\n⚠️ 警告: {len(zero_cost_orders)} 笔订单的成本为0")
            print("可能原因:")
            print("1. 产品ID未匹配到成本数据")
            print("2. 产品数量解析失败")
            print("3. 成本数据缺失")
            print("前5笔零成本订单:")
            print(zero_cost_orders[['Order_ID', 'Product_ID']].head())
        
        negative_profit_orders = detailed_df[detailed_df['Profit'] < 0]
        if len(negative_profit_orders) > 0:
            print(f"\n⚠️ 警告: {len(negative_profit_orders)} 笔订单的利润为负")
            print("前5笔负利润订单:")
            print(negative_profit_orders[['Order_ID', 'Total_amount', 'Total_Cost', 'Profit']].head())
        
        # 检查单个产品订单的成本计算
        single_product_orders = detailed_df[detailed_df['Product_ID'].apply(lambda x: ',' not in str(x))]
        if not single_product_orders.empty:
            print("\n单个产品订单的成本计算验证:")
            for idx, row in single_product_orders.head(5).iterrows():
                print(f"订单ID: {row['Order_ID']}")
                print(f"产品ID: {row['Product_ID']}")
                print(f"总金额: {row['Total_amount']}")
                print(f"总成本: {row['Total_Cost']}")
                print(f"利润: {row['Profit']}")
                print()
        
    except Exception as e:
        print(f"\n❌ 处理过程中出错: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\n💡 建议检查:")
        print("1. 确保输入文件路径正确")
        print("2. 确认Trello导出文件包含'Product_ID'列")
        print("3. 确认成本文件包含'Product_ID'和'Cost'列")
        print("4. 检查列名是否匹配（注意大小写和空格）")
        
        # 尝试显示文件列名
        try:
            df = pd.read_csv(trello_csv, nrows=1)
            print(f"\nTrello文件列名: {', '.join(df.columns)}")
        except:
            pass
        
        try:
            df = pd.read_excel(cost_excel, nrows=1)
            print(f"成本文件列名: {', '.join(df.columns)}")
        except:
            pass

订单利润计算工具 (终极修复版)

标准化函数测试:
  ✗ 输入: 'Seta_Tea-1F'
    期望: 'seta_tea_1f'
    实际: 'seta_tea-1f'

  ✓ 输入: 'Green Tea (Premium)'
    期望: 'green_tea_premium'
    实际: 'green_tea_premium'

  ✗ 输入: 'Milk_Tea*3'
    期望: 'milk_tea_3'
    实际: 'milk_tea3'

  ✗ 输入: 'Black Tea - 500g'
    期望: 'black_tea_500g'
    实际: 'black_tea_-_500g'

  ✓ 输入: 'Café au Lait'
    期望: 'cafe_au_lait'
    实际: 'cafe_au_lait'

  ✗ 输入: '产品编号_123'
    期望: 'chan_pin_bian_hao_123'
    实际: '_123'

  ✓ 输入: 'SET ITEM 1'
    期望: 'set_item_1'
    实际: 'set_item_1'

读取Trello数据: C:\Users\chine\Desktop\trello_project\分组聚合报表_20250812_132825.csv

❌ 处理过程中出错: [Errno 2] No such file or directory: 'C:\\Users\\chine\\Desktop\\trello_project\\分组聚合报表_20250812_132825.csv'

💡 建议检查:
1. 确保输入文件路径正确
2. 确认Trello导出文件包含'Product_ID'列
3. 确认成本文件包含'Product_ID'和'Cost'列
4. 检查列名是否匹配（注意大小写和空格）


Traceback (most recent call last):
  File "C:\Users\MI\AppData\Local\Temp\ipykernel_42664\2695265071.py", line 287, in <module>
    output_file, final_df, detailed_df = merge_and_calculate_profit(trello_csv, cost_excel)
                                         ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\MI\AppData\Local\Temp\ipykernel_42664\2695265071.py", line 71, in merge_and_calculate_profit
    df_trello = pd.read_csv(trello_csv_path)
  File "C:\Users\MI\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\io\parsers\readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "C:\Users\MI\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\io\parsers\readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "C:\Users\MI\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\io\parsers\readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, se

In [3]:
import pandas as pd
import re
from datetime import datetime
import os
import numpy as np
from collections import defaultdict
import unicodedata

def normalize_string(s):
    """标准化字符串：移除特殊字符、转换为小写、移除空格"""
    if not isinstance(s, str):
        s = str(s)
    # 移除重音符号
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    # 转换为小写
    s = s.lower()
    # 移除所有非字母数字字符（除了下划线和连字符）
    s = re.sub(r'[^\w\s-]', '', s)
    # 替换多个空格为单个空格
    s = re.sub(r'\s+', ' ', s)
    # 移除首尾空格
    s = s.strip()
    # 替换空格为下划线
    s = re.sub(r'\s', '_', s)
    return s

def parse_product_id(product_id):
    """
    更健壮的产品ID解析函数
    支持多种格式：基础产品ID+数量、基础产品ID(数量)、基础产品ID*数量
    例如：'seta_tea_1f67', 'seta_tea_1f(67)', 'seta_tea_1f*67'
    """
    original_id = str(product_id).strip()
    if not original_id:
        return original_id, 1
    
    # 尝试匹配数字后缀格式
    match = re.search(r'^(.+?)(\d{1,10})$', original_id)
    if match:
        base_id = match.group(1).rstrip('_')  # 移除末尾可能的下划线
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配括号格式
    match = re.search(r'^(.+?)\((\d{1,10})\)$', original_id)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配乘号格式
    match = re.search(r'^(.+?)\*(\d{1,10})$', original_id)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 尝试匹配x格式
    match = re.search(r'^(.+?)[x×](\d{1,10})$', original_id, re.IGNORECASE)
    if match:
        base_id = match.group(1).strip()
        quantity = int(match.group(2))
        return base_id, quantity
    
    # 默认返回产品ID和数量1
    return original_id, 1

def merge_and_calculate_profit(trello_csv_path, cost_excel_path):
    # 读取Trello导出的CSV文件
    print(f"读取Trello数据: {trello_csv_path}")
    df_trello = pd.read_csv(trello_csv_path)
    
    # 读取成本Excel文件
    print(f"读取成本数据: {cost_excel_path}")
    df_cost = pd.read_excel(cost_excel_path)
    
    # 打印初始数据预览
    print("\nTrello数据预览:")
    print(df_trello.head(3))
    print("\n成本数据预览:")
    print(df_cost.head(3))
    
    # 数据预处理
    # 确保金额列是数值类型
    df_trello['Total_amount'] = pd.to_numeric(df_trello['Total_amount'], errors='coerce').fillna(0)
    
    print("\nTrello文件列名:", df_trello.columns.tolist())
    print("成本文件列名:", df_cost.columns.tolist())
    
    # 处理多个产品的情况
    # 检查Product_ID列是否存在
    if 'Product_ID' not in df_trello.columns:
        # 尝试找到可能的替代列名
        possible_names = ['产品ID', 'product_id', '商品ID', '产品编号', 'productid', '商品编号']
        for name in possible_names:
            if name in df_trello.columns:
                df_trello.rename(columns={name: 'Product_ID'}, inplace=True)
                print(f"已将列 '{name}' 重命名为 'Product_ID'")
                break
        else:
            raise KeyError("找不到产品ID列，请确保数据包含'Product_ID'列")
    
    # 标准化产品ID列名
    if 'Product_ID' not in df_cost.columns:
        # 尝试找到可能的替代列名
        possible_names = ['产品ID', 'product_id', '商品ID', '产品编号', 'productid', '商品编号']
        for name in possible_names:
            if name in df_cost.columns:
                df_cost.rename(columns={name: 'Product_ID'}, inplace=True)
                print(f"已将成本文件列 '{name}' 重命名为 'Product_ID'")
                break
        else:
            raise KeyError("成本文件中找不到产品ID列")
    
    # 创建原始订单的唯一标识符 - 使用索引
    df_trello['order_id_temp'] = df_trello.index
    
    # 拆分Product_ID列（如果有多个产品）
    df_trello['Product_IDs'] = df_trello['Product_ID'].astype(str).str.split(',')
    
    # 将每个产品拆分为单独的行
    df_exploded = df_trello.explode('Product_IDs')
    df_exploded['Product_IDs'] = df_exploded['Product_IDs'].str.strip()
    
    # 解析产品ID和数量
    print("\n开始解析产品ID和数量...")
    parsed_info = df_exploded['Product_IDs'].apply(parse_product_id)
    df_exploded['Base_Product_ID'] = parsed_info.apply(lambda x: x[0])
    df_exploded['Quantity'] = parsed_info.apply(lambda x: x[1])
    
    # 标准化所有ID以进行匹配
    print("\n标准化产品ID以进行匹配...")
    df_exploded['Base_Product_ID_Norm'] = df_exploded['Base_Product_ID'].apply(normalize_string)
    df_cost['Product_ID_Norm'] = df_cost['Product_ID'].apply(normalize_string)
    
    # 创建成本映射字典
    cost_map = defaultdict(float)
    for _, row in df_cost.iterrows():
        norm_id = row['Product_ID_Norm']
        cost = float(row['Cost']) if not pd.isna(row['Cost']) else 0
        cost_map[norm_id] = cost
    
    # 计算每个产品的总成本
    print("\n计算每个产品的成本...")
    df_exploded['Product_Cost'] = df_exploded.apply(lambda row: 
        cost_map.get(row['Base_Product_ID_Norm'], 0) * row['Quantity'], axis=1)
    
    # 统计未匹配的产品
    unmatched_df = df_exploded[df_exploded['Product_Cost'] == 0]
    unmatched_products = unmatched_df['Base_Product_ID'].unique()
    
    if len(unmatched_products) > 0:
        print(f"\n⚠️ 警告: {len(unmatched_products)} 种产品未匹配到成本:")
        print(f"未匹配产品示例 ({min(5, len(unmatched_products))} of {len(unmatched_products)}):")
        for product in unmatched_products[:5]:
            norm_product = normalize_string(product)
            print(f"  - 原始ID: '{product}'")
            print(f"    标准化ID: '{norm_product}'")
            print(f"    在成本文件中匹配: {'是' if norm_product in cost_map else '否'}")
            
            # 查找最相似的ID
            similarities = []
            for cost_id in cost_map.keys():
                if norm_product in cost_id or cost_id in norm_product:
                    similarity = len(set(norm_product) & set(cost_id)) / max(len(norm_product), len(cost_id))
                    similarities.append((cost_id, similarity))
            
            if similarities:
                best_match = max(similarities, key=lambda x: x[1])
                print(f"    最相似的成本ID: '{best_match[0]}' (相似度: {best_match[1]:.2f})")
            else:
                print("    没有找到相似的ID")
    
    # 按原始订单分组计算总成本
    print("\n按订单分组计算总成本...")
    order_costs = df_exploded.groupby('order_id_temp')['Product_Cost'].sum().reset_index()
    order_costs.rename(columns={'Product_Cost': 'Total_Cost'}, inplace=True)
    
    # 合并回原始数据
    result_df = pd.merge(
        df_trello.drop(columns=['Product_IDs']),
        order_costs,
        on='order_id_temp',
        how='left'
    )
    
    # 填充缺失的成本为0
    result_df['Total_Cost'] = result_df['Total_Cost'].fillna(0)
    
    # 计算利润
    result_df['Profit'] = result_df['Total_amount'] - result_df['Total_Cost']
    
    # 生成Order_ID
    def generate_order_id(row):
        try:
            # 格式化日期：从YYYY-MM-DD转换为YYYYMMDD
            date_str = str(row['Date'])
            date_part = ''.join(filter(str.isdigit, date_str))[:8]
            
            # 获取List_ID的最后4位
            list_id = str(row['List_ID'])
            list_id_part = list_id[-4:] if len(list_id) >= 4 else list_id.zfill(4)
            
            return f"ORD-V9-{date_part}-{list_id_part}"
        except Exception as e:
            print(f"生成Order_ID时出错: {str(e)}")
            return "ORD-ERROR"
    
    print("\n生成订单ID...")
    result_df['Order_ID'] = result_df.apply(generate_order_id, axis=1)
    
    # 只保留指定的列并按指定顺序排列
    final_columns = ['order_id_temp', 'Order_ID', 'Date', 'Total_Cost', 'Profit', 'Total_amount', 'Phone']
    
    # 确保所有列都存在
    available_columns = [col for col in final_columns if col in result_df.columns]
    missing_columns = set(final_columns) - set(available_columns)
    
    if missing_columns:
        print(f"⚠️ 缺少列: {', '.join(missing_columns)}")
        # 添加缺失列
        for col in missing_columns:
            result_df[col] = None
    
    final_df = result_df[final_columns]
    
    # 生成输出文件名
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"Profit_Report_{timestamp}.csv"
    output_path = os.path.join(os.path.dirname(trello_csv_path), output_filename)
    
    # 保存为CSV
    final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"✅ 利润报告已保存至: {output_path}")
    
    # 保存详细数据供参考
    detailed_filename = os.path.join(os.path.dirname(trello_csv_path), f"Detailed_Report_{timestamp}.csv")
    result_df.to_csv(detailed_filename, index=False, encoding='utf-8-sig')
    print(f"✅ 详细报告已保存至: {detailed_filename}")
    
    # 保存未匹配产品数据
    if not unmatched_df.empty:
        unmatched_filename = os.path.join(os.path.dirname(trello_csv_path), f"Unmatched_Products_{timestamp}.csv")
        unmatched_df[['order_id_temp', 'Product_IDs', 'Base_Product_ID', 'Base_Product_ID_Norm', 'Quantity']].to_csv(unmatched_filename, index=False)
        print(f"⚠️ 未匹配产品数据已保存至: {unmatched_filename}")
    
    return output_path, final_df, result_df

if __name__ == "__main__":
    # 文件路径
    trello_csv = r"C:\Users\MI\Desktop\desktop\trello_project\2026-Trello-data.csv"
    cost_excel = r"C:\Users\MI\Desktop\desktop\trello_project\Cost.xlsx"
    
    try:
        print("="*50)
        print("订单利润计算工具 (按指定顺序输出)")
        print("="*50)
        
        # 处理文件
        output_file, final_df, detailed_df = merge_and_calculate_profit(trello_csv, cost_excel)
        
        print(f"\n✅ 文件处理完成！输出文件: {output_file}")
        print(f"📊 订单数量: {len(final_df)}")
        
        # 显示前5行预览
        print("\n最终报告预览:")
        print(final_df.head())
        
        # 计算总利润
        total_profit = final_df['Profit'].sum()
        total_revenue = final_df['Total_amount'].sum()
        total_cost = final_df['Total_Cost'].sum()
        
        print("\n汇总统计:")
        print(f"总收入: ¥{total_revenue:.2f}")
        print(f"总成本: ¥{total_cost:.2f}")
        print(f"总利润: ¥{total_profit:.2f}")
        print(f"平均利润率: {(total_profit/total_revenue*100 if total_revenue != 0 else 0):.2f}%")
        
    except Exception as e:
        print(f"\n❌ 处理过程中出错: {str(e)}")
        import traceback
        traceback.print_exc()
        print("\n💡 建议检查:")
        print("1. 确保输入文件路径正确")
        print("2. 确认Trello导出文件包含'Product_ID'列")
        print("3. 确认成本文件包含'Product_ID'和'Cost'列")
        print("4. 检查列名是否匹配（注意大小写和空格）")
        
        # 尝试显示文件列名
        try:
            df = pd.read_csv(trello_csv, nrows=1)
            print(f"\nTrello文件列名: {', '.join(df.columns)}")
        except:
            pass
        
        try:
            df = pd.read_excel(cost_excel, nrows=1)
            print(f"成本文件列名: {', '.join(df.columns)}")
        except:
            pass

订单利润计算工具 (按指定顺序输出)
读取Trello数据: C:\Users\MI\Desktop\desktop\trello_project\2026-Trello-data.csv
读取成本数据: C:\Users\MI\Desktop\desktop\trello_project\Cost.xlsx

Trello数据预览:
           List_Name        Date                  Product_ID  Total_amount  \
0  1.15 Terry Cheang  2025-01-15  1_fortune_c:30,Keychain:35         890.0   
1    1.15 アイコ 30份+29  2025-01-25                JP:30,JP2:29         720.0   
2     1.23 Amy 21份牛皮  2025-01-23                     1_5c:21         184.0   

   TEL  Discount_Rate  Final_Price  類別  
0  NaN              0        890.0 NaN  
1  NaN              0        720.0 NaN  
2  NaN              0        184.0 NaN  

成本数据预览:
  Unnamed: 0    Product_ID   Product_Name  Cost  profit  selling price
0        NaN   seta_tea_1f  Set-A ( 茶+幸 )   5.2     NaN           14.8
1        NaN  setb_coff_5c  Set-B ( 啡+迷 )   9.9     NaN           19.8
2        NaN   setc_tea_5c  Set-C ( 茶+迷 )   8.5     NaN           15.6

Trello文件列名: ['List_Name', 'Date', 'Product_ID', 'Total_amoun